# Multimodal TACS (M-TACS) - SUN397, Offline Training

This is an Internet-disabled training copy based on `mtacs_sun397_visual_text_context.ipynb`. It preserves the TACS source notebook and the existing API-based notebook. The SUN397 split, candidate pool, DINOv3 ViT-S/16 selector, full-pool scoring, policy/reward logic, optimizer, scheduler, visual transforms, evaluation, and checkpoint workflow remain from the current M-TACS implementation.

The preprocessing pipeline is deliberately completed before training:

`prepared captions.csv -> frozen CLIP text_embeddings.pt (or supplied cache) -> train M-TACS`

The caption CSV is already prepared. This notebook does not call a VLM API, load Florence-2, install packages, or download model files. It validates the CSV and either reuses a supplied CLIP embedding cache or encodes captions with CLIP files attached locally to Kaggle. The visual TACS selector remains image-only; text is consumed only by the downstream branch.

`EXPERIMENT_MODE` supports `no_context`, `original_tacs`, `text_only`, `mtacs`, and `mtacs_shuffled`. Use the same captions and embedding cache across separate ablation runs. Validation chooses the checkpoint; official test metrics are calculated in the final evaluation cell.

**Reward:** M-TACS compares query-plus-caption against query-plus-selected-image-plus-the-same-caption. Only visual context changes between branches. Original TACS retains the source notebook reward. Text enters after pooled visual representations through a trainable projection and gate; the selector and TACS policy are not given captions, CLIP embeddings, or labels.


## Offline preprocessing order

1. Turn Kaggle Internet off and attach SUN397, DINOv3 code/weights, and the completed `captions.csv` as Kaggle inputs.
2. No Florence-2 model or VLM API is used in this notebook. The CSV is copied from Kaggle Input to the writable output folder automatically when needed.
3. For CLIP, either attach a matching `text_embeddings.pt` cache, or attach the local `openai/clip-vit-base-patch32` model files and set `CLIP_MODEL_LOCAL_PATH`. The notebook never fetches model files.
4. Run the cells in order. CLIP embeddings are cached before training and CLIP is unloaded; training then uses only the cached embeddings.

Required Kaggle inputs: SUN397 images/partition files, local DINOv3 repository and weights, and `captions.csv`. If no matching `text_embeddings.pt` is supplied, also attach the local CLIP model directory and set `CLIP_MODEL_LOCAL_PATH`; Transformers must already be installed in the Kaggle image. Do not change the seed or TACS candidate-pool settings between ablations.


In [19]:
# Offline-only setup: this notebook deliberately does not run pip or download packages.
import importlib.util
_required_modules = {"numpy": "numpy", "pandas": "pandas", "torch": "torch", "torchvision": "torchvision", "PIL": "Pillow", "matplotlib": "matplotlib", "tqdm": "tqdm", "sklearn": "scikit-learn"}
_missing_modules = [package for module, package in _required_modules.items() if importlib.util.find_spec(module) is None]
if _missing_modules:
    raise RuntimeError("Missing preinstalled Kaggle packages (Internet is disabled): " + ", ".join(_missing_modules))
print("Offline package check passed; no package installation or Internet access is used.")


Offline package check passed; no package installation or Internet access is used.


In [20]:
# === M-TACS: CONFIGURATION ===
import os
import json
import random
import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

EXPERIMENT_MODE = "mtacs"  # no_context | original_tacs | text_only | mtacs | mtacs_shuffled
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
CLIP_MODEL_LOCAL_PATH = "/kaggle/input/datasets/nguyenducxu/vit-32/model_clip_vit"  # Set to the attached local CLIP model folder if no text_embeddings.pt cache is attached.
CAPTIONS_INPUT_PATH = "/kaggle/input/datasets/nguyenducxu/captions-csv/captions (3).csv"  # Optional: set to the exact /kaggle/input/.../captions.csv path if auto-discovery misses it.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
VLM_MODEL = "microsoft/Florence-2-base"
PROMPT_VERSION = "florence2_detailed_v1"
CAPTION_TASK_PROMPT = "<MORE_DETAILED_CAPTION>"
USE_CACHED_CAPTIONS = True
USE_CACHED_TEXT_EMBEDDINGS = True
ENABLE_LEAKAGE_CHECK = True
CAPTION_MAX_IMAGES_PER_SPLIT = None  # Keep None for the complete benchmark dataset.
RUN_ABLATIONS_IN_SEPARATE_RUNS = True
LAMBDA_POLICY = float(globals().get("TACS_POLICY_WEIGHT", 1.0))

PROMPT_TEXT = "Florence-2 task <MORE_DETAILED_CAPTION>; retain 5-8 readable visual-attribute words and reject SUN397 class-name leakage."
print("Experiment mode:", EXPERIMENT_MODE)
print("Caption model:", VLM_MODEL)
print("Frozen CLIP model:", CLIP_MODEL_NAME)


Experiment mode: mtacs
Caption model: microsoft/Florence-2-base
Frozen CLIP model: openai/clip-vit-base-patch32


## SUN397 setup and original candidate-pool construction

The following cell is retained from the source notebook; only the output directory and stable image IDs were added.

In [21]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

torch.backends.cudnn.benchmark = True
# Tận dụng Tensor Cores thế hệ mới trên dòng RTX / GPU Ampere+
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

# --- ĐƯỜNG DẪN OUTPUT (Ghi vào working của Kaggle) ---
PROJECT_DIR = Path('/kaggle/working')
OUT_PUT_DIR = PROJECT_DIR / 'outputs' / 'mtacs_sun397_visual_text_context'
OUTPUT_DIR = OUT_PUT_DIR
OUT_PUT_DIR.mkdir(parents=True, exist_ok=True)

# --- ĐƯỜNG DẪN DATASET (SUN397) ---
DATASET_ROOT = Path('/kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397/SUN397')
if not DATASET_ROOT.exists():
    fallback_candidates = [
        Path('/kaggle/input/sun397/SUN397/SUN397'),
        Path('/kaggle/input/sun397/SUN397'),
        Path('/kaggle/input/sun397'),
        Path('/kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397'),
    ]
    for candidate in fallback_candidates:
        if candidate.exists():
            DATASET_ROOT = candidate
            break

IMAGE_ROOT = DATASET_ROOT

# --- ĐƯỜNG DẪN DINOv3 REPO & WEIGHTS ---
DINOV3_REPO_DIR = '/kaggle/input/datasets/nguyenducxu/dinov3-code/dinov3-main'
if not Path(DINOV3_REPO_DIR).exists():
    for p in [Path('/kaggle/input/dinov3-code/dinov3-main'), Path('/kaggle/input/dinov3-code')]:
        if p.exists():
            DINOV3_REPO_DIR = str(p)
            break

DINOV3_MODEL_NAME = 'dinov3_vits16'
DINOV3_WEIGHTS = '/kaggle/input/datasets/nguyenducxu/dinov3/dinov3_vits16_pretrain_lvd1689m-08c60483.pth'
if not Path(DINOV3_WEIGHTS).exists():
    for p in [
        Path('/kaggle/input/dinov3/dinov3_vits16_pretrain_lvd1689m-08c60483.pth'),
        Path('/kaggle/input/dinov3/dinov3_vits16.pth'),
    ]:
        if p.exists():
            DINOV3_WEIGHTS = str(p)
            break

BACKBONE_NAME = DINOV3_MODEL_NAME
FEATURE_DIM = 384

IMAGE_SIZE = 384
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
BACKBONE_LR = 3e-6
HEAD_LR = 1e-4
BACKBONE_WEIGHT_DECAY = 0.05
HEAD_WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
TOKEN_HEAD_DROPOUT = 0.3
FREEZE_BACKBONE_EPOCHS = 0
WARMUP_EPOCHS = 5
PATIENCE = 10
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 8
NUM_CLASSES = 397  # SUN397 có 397 scene categories
TOP_K = 5
CONTEXT_TOP_K = 1
RETRIEVAL_SCORE_WEIGHT = 0.0
SUPCON_WEIGHT = 0.05
SUPCON_TEMPERATURE = 0.10
USE_BBOX_CROP = False  # SUN397 là bài toán scene classification, không dùng crop bbox
CANDIDATE_POOL_FRACTION = 0.20
VAL_FRACTION = 0.10
RETRIEVAL_BATCH_SIZE = 512
RETRIEVAL_CHUNK_SIZE = 4096
QUERY_CONTEXT_GATE_DROPOUT = 0.20
QUERY_CONTEXT_GATE_BIAS = -0.7
EVAL_TTA_HORIZONTAL_FLIP = False
USE_CONTEXT_LABEL_PRIOR = False
CONTEXT_LABEL_PRIOR_WEIGHT = 0.25
CONTEXT_LABEL_PRIOR_TEMPERATURE = 0.07

TRAIN_ON_FULL_TRAIN = False
FULL_TRAIN_MONITOR_FRACTION = 0.05  # chi dung khi TRAIN_ON_FULL_TRAIN=True

# --- TACS selector hyperparameters ---
TACS_CANDIDATES_PER_QUERY = None  # None nghĩa là dùng toàn bộ candidate pool
TACS_PREFILTER_TOP_SIMILAR = 0
TACS_GUMBEL_TAU = 0.10
TACS_POLICY_WEIGHT = 1.0
TACS_ENTROPY_WEIGHT = 0.01
TACS_SELECTOR_LR = 1e-4
TACS_SELECTOR_WEIGHT_DECAY = 1e-4
TACS_SELECTOR_DROPOUT = 0.10
TACS_SELECTOR_TEMPERATURE = 1.0

# --- Full-pool selector memory controls ---
TACS_USE_FULL_CANDIDATE_POOL = True
CANDIDATE_IMAGE_BANK_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CANDIDATE_IMAGE_BANK_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
CANDIDATE_FEATURE_BANK_BATCH_SIZE = 128
FULL_POOL_SCORE_CHUNK_SIZE = 1024

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 48, IMAGE_SIZE + 48)),
    transforms.RandomResizedCrop(
        (IMAGE_SIZE, IMAGE_SIZE),
        scale=(0.60, 1.0),
        ratio=(0.80, 1.25),
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.25,
            contrast=0.25,
            saturation=0.20,
            hue=0.04,
        )
    ], p=0.75),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))
    ], p=0.15),
    transforms.ToTensor(),
    transforms.RandomErasing(
        p=0.20,
        scale=(0.02, 0.12),
        ratio=(0.3, 3.3),
        value="random",
    ),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

_mean = globals().get("DINO_MEAN", globals().get("IMAGENET_MEAN", (0.485, 0.456, 0.406)))
_std = globals().get("DINO_STD", globals().get("IMAGENET_STD", (0.229, 0.224, 0.225)))

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=_mean, std=_std),
])


def load_class_names(dataset_root):
    classes_path = dataset_root / 'ClassName.txt'
    if not classes_path.is_file():
        fallback_paths = [
            dataset_root.parent / 'ClassName.txt',
            dataset_root / 'classes.txt',
            Path('/kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397/SUN397/ClassName.txt'),
            Path('/kaggle/input/sun397/SUN397/SUN397/ClassName.txt'),
        ]
        for p in fallback_paths:
            if p.is_file():
                classes_path = p
                break

    if not classes_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy ClassName.txt tại: {classes_path}")

    class_names = []
    with open(classes_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                # Dòng có dạng /a/abbey hoặc a/abbey -> chuẩn hóa bỏ dấu / ở đầu
                class_names.append(line.lstrip('/'))

    global NUM_CLASSES
    if len(class_names) != NUM_CLASSES:
        print(f"Cập nhật NUM_CLASSES từ {NUM_CLASSES} thành {len(class_names)}")
        NUM_CLASSES = len(class_names)
    return class_names


def load_sun397_annotations(dataset_root, image_root, class_names):
    class_to_id = {cls.lstrip('/'): idx for idx, cls in enumerate(class_names)}

    # 1. Tìm các file partition chuẩn (Training_01.txt / Testing_01.txt) nếu có sẵn trong dataset
    train_part_file, test_part_file = None, None
    search_dirs = [
        dataset_root,
        dataset_root.parent / 'sun397_other',
        dataset_root.parent / 'sun397_other' / 'Partitions',
        dataset_root.parent.parent / 'sun397_other',
        Path('/kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397/sun397_other'),
    ]
    for d in search_dirs:
        if d.is_dir():
            t1 = d / 'Training_01.txt'
            te1 = d / 'Testing_01.txt'
            if t1.is_file() and te1.is_file():
                train_part_file, test_part_file = t1, te1
                break
            found_t = list(d.glob('**/Training_01.txt'))
            if found_t:
                train_part_file = found_t[0]
                cand_te = train_part_file.parent / 'Testing_01.txt'
                if cand_te.is_file():
                    test_part_file = cand_te
                    break

    records = []
    if train_part_file and test_part_file:
        print(f"Sử dụng official partition: {train_part_file.name} và {test_part_file.name}")
        for split_label, p_file in [('train', train_part_file), ('test', test_part_file)]:
            with open(p_file, 'r', encoding='utf-8') as f:
                for line in f:
                    rel_p = line.strip().lstrip('/')
                    if not rel_p:
                        continue
                    class_folder = str(Path(rel_p).parent).replace('\\', '/').lstrip('/')
                    if class_folder not in class_to_id:
                        continue
                    class_id = class_to_id[class_folder]
                    full_p = image_root / rel_p
                    if not full_p.is_file():
                        if (dataset_root / rel_p).is_file():
                            full_p = dataset_root / rel_p
                        else:
                            continue
                    records.append({
                        'rel_path': rel_p,
                        'fname': Path(rel_p).name,
                        'image_path': str(full_p.resolve()),
                        'class_id': int(class_id),
                        'is_train': 1 if split_label == 'train' else 0,
                    })
        df = pd.DataFrame(records)
    else:
        print("Không thấy file partition, tự động quét thư mục ảnh theo ClassName.txt và chia split 50/50 chuẩn...")
        all_rows = []
        rng = np.random.default_rng(SEED)
        valid_exts = {'.jpg', '.jpeg', '.png', '.bmp'}

        for cls_name in tqdm(class_names, desc="Đang đọc các lớp SUN397"):
            cls_dir = image_root / cls_name
            if not cls_dir.is_dir():
                cls_dir = dataset_root / cls_name
            if not cls_dir.is_dir():
                continue

            cls_id = class_to_id[cls_name]
            img_paths = [p for p in cls_dir.iterdir() if p.is_file() and p.suffix.lower() in valid_exts]
            img_paths.sort()
            if not img_paths:
                continue

            n_total = len(img_paths)
            indices = np.arange(n_total)
            rng.shuffle(indices)

            # Benchmark chuẩn SUN397: 50 train / 50 test mỗi class nếu >= 100 ảnh, hoặc chia đôi
            if n_total >= 100:
                n_train = 50
                n_test = 50
                train_idx = set(indices[:n_train])
                test_idx = set(indices[n_train:n_train + n_test])
            else:
                n_train = max(1, n_total // 2)
                train_idx = set(indices[:n_train])
                test_idx = set(indices[n_train:])

            for idx, p in enumerate(img_paths):
                if idx in train_idx:
                    is_train = 1
                elif idx in test_idx:
                    is_train = 0
                else:
                    continue

                try:
                    rel_p = str(p.relative_to(image_root)).replace('\\', '/')
                except Exception:
                    rel_p = f"{cls_name}/{p.name}"

                all_rows.append({
                    'rel_path': rel_p,
                    'fname': p.name,
                    'image_path': str(p.resolve()),
                    'class_id': int(cls_id),
                    'is_train': is_train,
                })
        df = pd.DataFrame(all_rows)

    # Thêm các cột dummy BBox để tương thích 100% với Dataset class ở Cell 3 mà không gây lỗi
    df['image_id'] = np.arange(len(df))
    df['bbox_x1'] = 0
    df['bbox_y1'] = 0
    df['bbox_x2'] = 99999
    df['bbox_y2'] = 99999

    train_df = df[df['is_train'] == 1].copy().reset_index(drop=True)
    train_df['split'] = 'train'
    train_df['row_id'] = np.arange(len(train_df))

    test_df = df[df['is_train'] == 0].copy().reset_index(drop=True)
    test_df['split'] = 'test'
    test_df['row_id'] = np.arange(len(test_df))

    return train_df, test_df


def split_train_val_by_class(df, val_fraction=0.10, seed=42):
    if not 0 < val_fraction < 1:
        raise ValueError('val_fraction phải nằm trong khoảng (0, 1)')
    rng = np.random.default_rng(seed)
    val_indices = []
    for _, group in df.groupby('class_id', sort=True):
        group_indices = group.index.to_numpy()
        if len(group_indices) <= 1:
            continue
        n_val = max(1, int(round(len(group_indices) * val_fraction)))
        n_val = min(n_val, len(group_indices) - 1)
        val_indices.extend(rng.choice(group_indices, size=n_val, replace=False).tolist())
    val_indices = np.array(sorted(val_indices), dtype=int)
    train_split = df.drop(index=val_indices).reset_index(drop=True)
    val_split = df.loc[val_indices].reset_index(drop=True)
    return train_split, val_split


# --- Tải dữ liệu SUN397 ---
class_names = load_class_names(DATASET_ROOT)
full_train_df, test_df = load_sun397_annotations(DATASET_ROOT, IMAGE_ROOT, class_names)
train_df, val_df = split_train_val_by_class(full_train_df, VAL_FRACTION, SEED)

if TRAIN_ON_FULL_TRAIN:
    _, monitor_val_df = split_train_val_by_class(
        full_train_df,
        FULL_TRAIN_MONITOR_FRACTION,
        SEED,
    )
    train_df = full_train_df.reset_index(drop=True)
    val_df = monitor_val_df.reset_index(drop=True)

candidate_rng = np.random.default_rng(SEED)
if CANDIDATE_POOL_FRACTION >= 1.0:
    candidate_pool_positions = np.arange(len(train_df), dtype=int)
else:
    candidate_pool_size = max(1, int(round(len(train_df) * CANDIDATE_POOL_FRACTION)))
    candidate_pool_positions = np.sort(
        candidate_rng.choice(len(train_df), size=candidate_pool_size, replace=False)
    )

candidate_df = train_df.iloc[candidate_pool_positions].reset_index(drop=True)
candidate_df.to_csv(OUT_PUT_DIR / 'candidate_pool.csv', index=False)
val_df.to_csv(OUT_PUT_DIR / 'validation_split.csv', index=False)

print('DATASET_ROOT:', DATASET_ROOT.resolve())
print('IMAGE_ROOT:', IMAGE_ROOT.resolve())
print('OUT_PUT_DIR:', OUT_PUT_DIR.resolve())
print(f'Standard train images: {len(full_train_df):,}')
print(f'Actual train images: {len(train_df):,}')
print(f'Validation images: {len(val_df):,}')
print(f'Train on full train: {TRAIN_ON_FULL_TRAIN}')
print(f'Standard test images: {len(test_df):,}')
print(f'Candidate pool: {len(candidate_df):,}/{len(train_df):,} actual-train images ({CANDIDATE_POOL_FRACTION:.0%})')
print(f'TACS selected context per query: {CONTEXT_TOP_K}')
print(f'TACS uses full candidate pool during training: {TACS_USE_FULL_CANDIDATE_POOL}')
print(f'TACS Gumbel tau: {TACS_GUMBEL_TAU}, policy weight: {TACS_POLICY_WEIGHT}')
print(f'BBox crop enabled: {USE_BBOX_CROP}')
print(f'Train label range: {int(train_df.class_id.min())}..{int(train_df.class_id.max())}')
print(f'Test label range: {int(test_df.class_id.min())}..{int(test_df.class_id.max())}')
display(train_df.head())
display(val_df.head())
display(candidate_df.head())

# Stable image identity shared by caption and embedding caches.
candidate_df['image_id'] = candidate_df['rel_path'].astype(str).str.replace('\\', '/', regex=False)
for _split_df in (train_df, val_df, test_df):
    _split_df['image_id'] = _split_df['rel_path'].astype(str).str.replace('\\', '/', regex=False)


DEVICE: cuda
Sử dụng official partition: Training_01.txt và Testing_01.txt
DATASET_ROOT: /kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397/SUN397
IMAGE_ROOT: /kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397/SUN397
OUT_PUT_DIR: /kaggle/working/outputs/mtacs_sun397_visual_text_context
Standard train images: 19,850
Actual train images: 17,865
Validation images: 1,985
Train on full train: False
Standard test images: 19,850
Candidate pool: 3,573/17,865 actual-train images (20%)
TACS selected context per query: 1
TACS uses full candidate pool during training: True
TACS Gumbel tau: 0.1, policy weight: 1.0
BBox crop enabled: False
Train label range: 0..396
Test label range: 0..396


,rel_path,fname,image_path,class_id,is_train,image_id,bbox_x1,bbox_y1,bbox_x2,bbox_y2,split,row_id
0,a/abbey/sun_aqswjsnjlrfzzhiz.jpg,sun_aqswjsnjlrfzzhiz.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,0,0,0,99999,99999,train,0
1,a/abbey/sun_aaoktempcmudsvna.jpg,sun_aaoktempcmudsvna.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,1,0,0,99999,99999,train,1
2,a/abbey/sun_arxtvfkkycnqhfkj.jpg,sun_arxtvfkkycnqhfkj.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,2,0,0,99999,99999,train,2
3,a/abbey/sun_axcfsjajikefppey.jpg,sun_axcfsjajikefppey.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,3,0,0,99999,99999,train,3
4,a/abbey/sun_aizsoszlxxvbjhqd.jpg,sun_aizsoszlxxvbjhqd.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,5,0,0,99999,99999,train,5


,rel_path,fname,image_path,class_id,is_train,image_id,bbox_x1,bbox_y1,bbox_x2,bbox_y2,split,row_id
0,a/abbey/sun_arsnqaftitwcpsog.jpg,sun_arsnqaftitwcpsog.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,4,0,0,99999,99999,train,4
1,a/abbey/sun_axcwnadpsaegsupw.jpg,sun_axcwnadpsaegsupw.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,21,0,0,99999,99999,train,21
2,a/abbey/sun_adbutqsfuzkgmrwy.jpg,sun_adbutqsfuzkgmrwy.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,31,0,0,99999,99999,train,31
3,a/abbey/sun_akpgqjmkigtgyyhf.jpg,sun_akpgqjmkigtgyyhf.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,36,0,0,99999,99999,train,36
4,a/abbey/sun_asgeghboyugsatii.jpg,sun_asgeghboyugsatii.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,49,0,0,99999,99999,train,49


,rel_path,fname,image_path,class_id,is_train,image_id,bbox_x1,bbox_y1,bbox_x2,bbox_y2,split,row_id
0,a/abbey/sun_aakbdcgfpksytcwj.jpg,sun_aakbdcgfpksytcwj.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,9,0,0,99999,99999,train,9
1,a/abbey/sun_afuhbsqqwerjshqj.jpg,sun_afuhbsqqwerjshqj.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,17,0,0,99999,99999,train,17
2,a/abbey/sun_alqgvwyohunprcsh.jpg,sun_alqgvwyohunprcsh.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,20,0,0,99999,99999,train,20
3,a/abbey/sun_argijfrtajgytjrs.jpg,sun_argijfrtajgytjrs.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,22,0,0,99999,99999,train,22
4,a/abbey/sun_atbseznnhjnnmefq.jpg,sun_atbseznnhjnnmefq.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,27,0,0,99999,99999,train,27


## Offline caption cache and label-leakage audit

Caption generation is a separate preprocessing stage handled by `generate_sun397_captions.py`. This notebook only reads the completed CSV. It checks required columns, duplicate/missing image IDs, split alignment, word-count validity, and class-name leakage locally. Labels and class names are never model inputs.


In [22]:
# === M-TACS: OFFLINE CAPTION CACHE ===
CAPTION_CSV = OUT_PUT_DIR / "captions.csv"
TEXT_EMBEDDINGS_PATH = OUT_PUT_DIR / "text_embeddings.pt"
CAPTION_CONFIG_PATH = OUT_PUT_DIR / "caption_config.json"
LAMBDA_POLICY = float(TACS_POLICY_WEIGHT)

if not CAPTION_CSV.is_file():
    if CAPTIONS_INPUT_PATH is not None:
        caption_source = Path(CAPTIONS_INPUT_PATH)
        if not caption_source.is_file():
            raise FileNotFoundError(f"CAPTIONS_INPUT_PATH does not exist: {caption_source}")
        shutil.copy2(caption_source, CAPTION_CSV)
        print("Copied captions from Kaggle Input:", caption_source)
    else:
        uploaded_csvs = sorted(Path("/kaggle/input").glob("*/captions.csv"))
        if len(uploaded_csvs) == 1:
            shutil.copy2(uploaded_csvs[0], CAPTION_CSV)
            print("Copied captions from Kaggle Input:", uploaded_csvs[0])
        elif len(uploaded_csvs) > 1:
            raise RuntimeError("Multiple captions.csv files found. Set CAPTIONS_INPUT_PATH in the configuration cell. Found: " + str(uploaded_csvs))

def _normalize_phrase(text):
    text = str(text).replace("_", " ").replace("/", " ").replace("-", " ").lower()
    return re.sub(r"[^a-z0-9]+", " ", text).strip()


def make_forbidden_terms(names):
    terms = set()
    for name in names:
        normalized = _normalize_phrase(name)
        basename = _normalize_phrase(str(name).replace("\\", "/").split("/")[-1])
        if normalized:
            terms.add(normalized)
        if basename:
            terms.add(basename)
    return sorted(terms, key=lambda term: (-len(term.split()), -len(term)))


def validate_caption_no_leakage(caption, forbidden_terms):
    normalized = f" {_normalize_phrase(caption)} "
    count = len(normalized.split())
    matched = [term for term in forbidden_terms if f" {term} " in normalized]
    return {"valid": bool(5 <= count <= 8 and not matched), "leakage_detected": bool(matched), "matched_terms": matched}


def _as_bool(value):
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def all_caption_samples():
    frames = []
    for split_name, frame in (("train", train_df), ("val", val_df), ("test", test_df)):
        part = frame[["image_id", "image_path"]].copy()
        part["split"] = split_name
        frames.append(part)
    samples = pd.concat(frames, ignore_index=True).drop_duplicates("image_id").reset_index(drop=True)
    if CAPTION_MAX_IMAGES_PER_SPLIT is not None:
        samples = samples.groupby("split", group_keys=False).head(int(CAPTION_MAX_IMAGES_PER_SPLIT)).reset_index(drop=True)
    return samples


if not USE_CACHED_CAPTIONS:
    raise RuntimeError("This offline training notebook requires a completed captions.csv. Run generate_sun397_captions.py first.")
if not CAPTION_CSV.is_file():
    raise FileNotFoundError(
        f"Caption CSV not found at {CAPTION_CSV}. Attach it as a Kaggle Input, run `!find /kaggle/input -name captions.csv`, then set CAPTIONS_INPUT_PATH in the configuration cell to that exact path and rerun this cell."
    )

caption_df = pd.read_csv(CAPTION_CSV).fillna("")
required_caption_columns = {
    "image_id", "split", "final_caption", "caption_valid", "leakage_detected",
    "retry_count", "vlm_model", "prompt_version",
}
missing_columns = required_caption_columns - set(caption_df.columns)
if missing_columns:
    raise ValueError(f"captions.csv is missing required columns: {sorted(missing_columns)}")
caption_df = caption_df.drop_duplicates("image_id", keep="last").copy()

caption_samples = all_caption_samples()
expected_ids = set(caption_samples["image_id"].astype(str))
cached_ids = set(caption_df["image_id"].astype(str))
missing_ids = expected_ids - cached_ids
if missing_ids:
    raise RuntimeError(
        f"captions.csv is missing {len(missing_ids):,} dataset images. Resume the standalone caption script, then rerun this cell."
    )

# Keep only this run's images and use the notebook's authoritative split labels.
caption_df["image_id"] = caption_df["image_id"].astype(str)
caption_df = caption_samples[["image_id", "split"]].merge(
    caption_df.drop(columns=["split"], errors="ignore"), on="image_id", how="left", validate="one_to_one"
)
caption_models = sorted(set(caption_df["vlm_model"].astype(str)) - {"", "nan"})
if len(caption_models) == 1:
    VLM_MODEL = caption_models[0]
elif len(caption_models) > 1:
    VLM_MODEL = "mixed:" + ",".join(caption_models)
    print("Warning: captions.csv contains captions from multiple VLMs:", caption_models)
FORBIDDEN_TERMS = make_forbidden_terms(class_names) if ENABLE_LEAKAGE_CHECK else []
valid_flags = []
leak_flags = []
for row in caption_df.itertuples(index=False):
    checked = validate_caption_no_leakage(row.final_caption, FORBIDDEN_TERMS)
    leak = _as_bool(row.leakage_detected) or checked["leakage_detected"]
    valid = _as_bool(row.caption_valid) and checked["valid"] and not leak
    valid_flags.append(valid)
    leak_flags.append(leak)
caption_df["caption_valid"] = valid_flags
caption_df["leakage_detected"] = leak_flags

caption_df.to_csv(CAPTION_CSV, index=False)
CAPTION_CONFIG_PATH.write_text(json.dumps({
    "caption_model": VLM_MODEL,
    "prompt_version": PROMPT_VERSION,
    "caption_length": "5-8 words",
    "caption_task_prompt": CAPTION_TASK_PROMPT,
    "leakage_policy": "invalid captions receive a zero CLIP embedding",
    "clip_model": CLIP_MODEL_NAME,
}, indent=2), encoding="utf-8")
print("Caption rows:", len(caption_df))
print("Caption valid rate:", float(np.mean(valid_flags)))
print("Leakage detected rate:", float(np.mean(leak_flags)))
print("Caption models in cache:", caption_df["vlm_model"].value_counts(dropna=False).to_dict())
display(caption_df.head())


Caption rows: 39700
Caption valid rate: 0.602544080604534
Leakage detected rate: 0.3955415617128463
Caption models in cache: {'microsoft/Florence-2-base': 39700}


,image_id,split,final_caption,caption_valid,leakage_detected,retry_count,vlm_model,prompt_version
0,a/abbey/sun_aqswjsnjlrfzzhiz.jpg,train,The image shows a large red brick castle,False,True,0,microsoft/Florence-2-base,florence2_detailed_v1
1,a/abbey/sun_aaoktempcmudsvna.jpg,train,The image shows the ruins of an old,True,False,0,microsoft/Florence-2-base,florence2_detailed_v1
2,a/abbey/sun_arxtvfkkycnqhfkj.jpg,train,The image shows a woman standing in the,True,False,0,microsoft/Florence-2-base,florence2_detailed_v1
3,a/abbey/sun_axcfsjajikefppey.jpg,train,The image shows a large ornate cathedral with,True,False,0,microsoft/Florence-2-base,florence2_detailed_v1
4,a/abbey/sun_aizsoszlxxvbjhqd.jpg,train,The image shows a small white church with,True,False,0,microsoft/Florence-2-base,florence2_detailed_v1


## Frozen CLIP text embeddings, cached before training

If the cache matches the caption IDs/text/validity and CLIP model, it is reused without loading CLIP. Otherwise, the frozen CLIP text encoder runs once in batches and saves `text_embeddings.pt`. Invalid captions map to zero vectors and a false validity mask. The tokenizer and CLIP encoder are deleted and CUDA cache is cleared before the TACS model is constructed.


In [23]:
# === M-TACS: PRECOMPUTE / LOAD FROZEN CLIP TEXT EMBEDDINGS ===
import gc

def _caption_cache_inputs():
    caps = caption_df.drop_duplicates("image_id", keep="last").reset_index(drop=True)
    ids = caps["image_id"].astype(str).tolist()
    texts = caps["final_caption"].astype(str).tolist()
    valid = (
        caps["caption_valid"].map(_as_bool).to_numpy()
        & ~caps["leakage_detected"].map(_as_bool).to_numpy()
    )
    return caps, ids, texts, torch.as_tensor(valid, dtype=torch.bool)


caps_for_clip, caption_ids, caption_texts, caption_valid_mask = _caption_cache_inputs()
saved_cache = None
if USE_CACHED_TEXT_EMBEDDINGS:
    cache_paths = ([TEXT_EMBEDDINGS_PATH] if TEXT_EMBEDDINGS_PATH.is_file() else [])
    cache_paths += [p for p in sorted(Path("/kaggle/input").glob("*/text_embeddings.pt")) if p != TEXT_EMBEDDINGS_PATH]
    for cache_path in cache_paths:
        try:
            candidate_cache = torch.load(cache_path, map_location="cpu", weights_only=False)
            cache_matches = (
                candidate_cache.get("clip_model_name") == CLIP_MODEL_NAME
                and candidate_cache.get("image_ids") == caption_ids
                and candidate_cache.get("captions") == caption_texts
                and torch.equal(candidate_cache.get("valid_mask", torch.empty(0, dtype=torch.bool)), caption_valid_mask)
                and candidate_cache.get("embeddings") is not None
                and len(candidate_cache["embeddings"]) == len(caption_ids)
            )
        except Exception as exc:
            print(f"Skipping unreadable embedding cache {cache_path}: {type(exc).__name__}: {exc}")
            continue
        if cache_matches:
            saved_cache = candidate_cache
            if cache_path != TEXT_EMBEDDINGS_PATH:
                shutil.copy2(cache_path, TEXT_EMBEDDINGS_PATH)
            print("Reusing matching offline CLIP text embeddings:", cache_path)
            break


if saved_cache is None:
    clip_source = str(CLIP_MODEL_LOCAL_PATH) if CLIP_MODEL_LOCAL_PATH else CLIP_MODEL_NAME
    try:
        from transformers import CLIPTextModelWithProjection, CLIPTokenizer
        clip_tokenizer = CLIPTokenizer.from_pretrained(clip_source, local_files_only=True)
        clip_text_encoder = CLIPTextModelWithProjection.from_pretrained(clip_source, local_files_only=True).to(DEVICE).eval()
    except Exception as exc:
        raise RuntimeError("Offline CLIP assets unavailable. Attach a matching text_embeddings.pt, or attach the local CLIP model folder and set CLIP_MODEL_LOCAL_PATH in the configuration cell. No network downloads are attempted.") from exc
    for parameter in clip_text_encoder.parameters():
        parameter.requires_grad_(False)
    CLIP_EMBED_DIM = int(clip_text_encoder.config.projection_dim)

    @torch.inference_mode()
    def encode_captions_with_clip(captions, batch_size=128):
        vectors = []
        for start in tqdm(range(0, len(captions), batch_size), desc="precompute CLIP text embeddings"):
            batch_text = [str(value) if str(value).strip() else "" for value in captions[start:start + batch_size]]
            tokens = clip_tokenizer(
                batch_text,
                padding=True,
                truncation=True,
                max_length=clip_text_encoder.config.max_position_embeddings,
                return_tensors="pt",
            ).to(DEVICE)
            vectors.append(clip_text_encoder(**tokens).text_embeds.float().cpu())
        return torch.cat(vectors, dim=0) if vectors else torch.empty((0, CLIP_EMBED_DIM))

    text_vectors = encode_captions_with_clip(caption_texts).clone()  # Make a regular tensor before masking inference-mode output.
    text_vectors[~caption_valid_mask] = 0
    saved_cache = {
        "image_ids": caption_ids,
        "captions": caption_texts,
        "embeddings": text_vectors,
        "valid_mask": caption_valid_mask,
        "clip_model_name": CLIP_MODEL_NAME,
        "embedding_dim": CLIP_EMBED_DIM,
    }
    temp_cache_path = TEXT_EMBEDDINGS_PATH.with_suffix(TEXT_EMBEDDINGS_PATH.suffix + ".tmp")
    torch.save(saved_cache, temp_cache_path)
    temp_cache_path.replace(TEXT_EMBEDDINGS_PATH)

    del clip_text_encoder, clip_tokenizer, text_vectors
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    print("Precomputed CLIP; model/tokenizer unloaded and CUDA cache released.")
else:
    saved_cache = saved_cache

text_cache = saved_cache
CLIP_EMBED_DIM = int(text_cache["embedding_dim"])
text_embedding_lookup = {
    image_id: (text_cache["embeddings"][index].float(), bool(text_cache["valid_mask"][index]))
    for index, image_id in enumerate(text_cache["image_ids"])
}
caption_text_by_id = dict(zip(text_cache["image_ids"], text_cache["captions"]))
if not expected_ids.issubset(text_embedding_lookup):
    raise RuntimeError("CLIP embedding cache does not cover all query/candidate image IDs.")
print("Embedding cache:", TEXT_EMBEDDINGS_PATH)
print("Embedding tensor:", tuple(text_cache["embeddings"].shape), "| CLIP dim:", CLIP_EMBED_DIM)
print("CLIP encoder resident:", "clip_text_encoder" in globals())


Reusing matching offline CLIP text embeddings: /kaggle/working/outputs/mtacs_sun397_visual_text_context/text_embeddings.pt
Embedding cache: /kaggle/working/outputs/mtacs_sun397_visual_text_context/text_embeddings.pt
Embedding tensor: (39700, 512) | CLIP dim: 512
CLIP encoder resident: False


## Dataset and data loaders

The original query split and visual transforms are retained. Stable POSIX `image_id` values join each image to its precomputed CPU CLIP embedding; the training loop never tokenizes captions or invokes a text encoder.


In [24]:
# === M-TACS: DATASET WITH IMAGE IDS / TEXT LOOKUP ===
class SUN397TACSQueryDataset(Dataset):
    def __init__(self, annotations, transform, crop_bbox=False):
        self.annotations = annotations.reset_index(drop=True).copy()
        self.transform = transform
        self.crop_bbox = crop_bbox

    def __len__(self):
        return len(self.annotations)

    def _open_image(self, row):
        image = Image.open(row["image_path"]).convert("RGB")
        if self.crop_bbox:
            left = max(0, int(row["bbox_x1"]) - 1)
            top = max(0, int(row["bbox_y1"]) - 1)
            right = min(image.width, max(left + 1, int(row["bbox_x2"])))
            bottom = min(image.height, max(top + 1, int(row["bbox_y2"])))
            image = image.crop((left, top, right, bottom))
        return image

    def __getitem__(self, index):
        row = self.annotations.iloc[index]
        image_id = str(row["image_id"])
        if image_id not in text_embedding_lookup:
            raise KeyError(f"No cached CLIP embedding for image_id={image_id}")
        embedding, valid = text_embedding_lookup[image_id]
        return {
            "query_image": self.transform(self._open_image(row)),
            "label": torch.tensor(int(row["class_id"]), dtype=torch.long),
            "fname": str(row["fname"]),
            "row_id": torch.tensor(int(row["row_id"]), dtype=torch.long),
            "image_id": image_id,
            "text_embedding": embedding.clone().float(),
            "text_valid": torch.tensor(valid, dtype=torch.bool),
        }


def make_shuffled_text_lookup(cache, seed):
    # A cyclic derangement within each split ensures a sample never receives its own caption.
    rng = np.random.default_rng(seed)
    captions = pd.read_csv(CAPTION_CSV).fillna("").drop_duplicates("image_id", keep="last")
    embedding_by_id = {key: (cache["embeddings"][i], bool(cache["valid_mask"][i])) for i, key in enumerate(cache["image_ids"])}
    shuffled = {}
    for _, group in captions.groupby("split", sort=True):
        ids = group["image_id"].astype(str).tolist()
        if len(ids) < 2:
            shuffled.update({key: embedding_by_id[key] for key in ids})
            continue
        order = rng.permutation(len(ids))
        ordered_ids = [ids[i] for i in order]
        offset = int(rng.integers(1, len(ids)))
        for i, image_id in enumerate(ordered_ids):
            shuffled[image_id] = embedding_by_id[ordered_ids[(i + offset) % len(ids)]]
    return shuffled


shuffled_text_lookup = make_shuffled_text_lookup(text_cache, SEED)
train_dataset = SUN397TACSQueryDataset(train_df, train_transform, crop_bbox=USE_BBOX_CROP)
val_dataset = SUN397TACSQueryDataset(val_df, eval_transform, crop_bbox=USE_BBOX_CROP)
test_dataset = SUN397TACSQueryDataset(test_df, eval_transform, crop_bbox=USE_BBOX_CROP)
loader_kwargs = {"num_workers": NUM_WORKERS, "pin_memory": torch.cuda.is_available()}
if NUM_WORKERS > 0:
    loader_kwargs.update(persistent_workers=True, prefetch_factor=2)
train_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=train_generator, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
print("Dataset sizes:", len(train_dataset), len(val_dataset), len(test_dataset))


Dataset sizes: 17865 1985 19850


## Original TACS model plus M-TACS downstream branch

The class definitions, candidate image bank, full-pool selector, feature refresh, and optimizer groups below are reused from the source notebook. The appended subclass adds only text projection, gate, and classification head.

In [27]:
from pathlib import Path
import gc
import torch


def resolve_dinov3_weights():
    path = Path(DINOV3_WEIGHTS)
    if not path.is_file():
        raise FileNotFoundError(f'Không tìm thấy file weight: {path}')
    return str(path)


def load_dinov3_model(repo_dir, model_name, weights_path):
    repo_dir = Path(repo_dir)
    if not repo_dir.is_dir():
        raise FileNotFoundError(f'Không tìm thấy thư mục mã nguồn: {repo_dir}')
    weights_path = Path(weights_path)
    if not weights_path.is_file():
        raise FileNotFoundError(f'Không tìm thấy file weight: {weights_path}')
    return torch.hub.load(str(repo_dir), model_name, source='local', weights=str(weights_path))


def load_dinov3_backbone():
    return load_dinov3_model(DINOV3_REPO_DIR, DINOV3_MODEL_NAME, resolve_dinov3_weights())


def ensure_dinov3_assets():
    resolve_dinov3_weights()
    repo_dir = Path(DINOV3_REPO_DIR)
    if not repo_dir.is_dir():
        raise FileNotFoundError(f'Không tìm thấy thư mục mã nguồn: {repo_dir}')
    print('DINOv3 assets đã sẵn sàng từ Kaggle Input.')

class SimpleImageDataset(Dataset):
    def __init__(self, annotations, transform, crop_bbox=False):
        self.annotations = annotations.reset_index(drop=True).copy()
        self.transform = transform
        self.crop_bbox = bool(crop_bbox)

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        row = self.annotations.iloc[index]
        with Image.open(row['image_path']) as source:
            image = source.convert('RGB')
        bbox_columns = {'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2'}
        if self.crop_bbox and bbox_columns.issubset(self.annotations.columns):
            left = max(0, int(row['bbox_x1']) - 1)
            top = max(0, int(row['bbox_y1']) - 1)
            right = min(image.width, max(left + 1, int(row['bbox_x2'])))
            bottom = min(image.height, max(top + 1, int(row['bbox_y2'])))
            image = image.crop((left, top, right, bottom))
        return self.transform(image), int(row['class_id']), str(row['fname']), int(row.get('row_id', index))


def extract_cls_features(backbone, images):
    features = backbone(images)
    if isinstance(features, dict):
        if 'x_norm_clstoken' in features:
            return features['x_norm_clstoken']
        if 'x_norm_cls_token' in features:
            return features['x_norm_cls_token']
        if 'x_norm_patchtokens' in features:
            return features['x_norm_patchtokens'].mean(dim=1)
        raise KeyError(f'Khong tim thay CLS token trong output keys: {list(features)}')
    return features[:, 0] if torch.is_tensor(features) and features.ndim == 3 else features


def extract_frozen_dino_embeddings(annotations, backbone, desc):
    dataset = SimpleImageDataset(annotations, eval_transform, crop_bbox=USE_BBOX_CROP)
    loader = DataLoader(dataset, batch_size=RETRIEVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,persistent_workers=True if NUM_WORKERS > 0 else False)
    embeddings = []
    backbone.eval()
    for parameter in backbone.parameters():
        parameter.requires_grad_(False)
    with torch.inference_mode():
        for images, *_ in tqdm(loader, desc=desc, leave=False):
            images = images.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=DEVICE.type == 'cuda'):
                features = extract_cls_features(backbone, images)
            features = F.normalize(features.float(), dim=1)
            embeddings.append(features.cpu())
    return torch.cat(embeddings, dim=0)




class DinoV3FrozenSimilarityContextClassifier(nn.Module):
    def __init__(self, repo_dir, model_name, weights_path, num_classes, feature_dim=384, allow_feature_fallback=False):
        super().__init__()
        self.backbone = load_dinov3_model(repo_dir, model_name, weights_path)
        self.feature_dim = feature_dim
        self.allow_feature_fallback = allow_feature_fallback
        self.active_fusion_mode = 'topk_token_attention'
        self.context_proj = nn.Linear(feature_dim, feature_dim)
        self.query_proj = nn.Linear(feature_dim, feature_dim)
        self.context_attention = nn.Linear(feature_dim, 1)
        self.fusion_norm = nn.LayerNorm(feature_dim)
        self.token_head = nn.Sequential(nn.Dropout(p=TOKEN_HEAD_DROPOUT), nn.Linear(feature_dim, num_classes))
        self.feature_fusion = nn.Sequential(nn.Linear(feature_dim * 2, feature_dim), nn.GELU(), nn.Dropout(p=TOKEN_HEAD_DROPOUT), nn.Linear(feature_dim, num_classes))
        for param in self.backbone.parameters():
            param.requires_grad_(False)

    def _prefix_token_count(self):
        if hasattr(self.backbone, 'num_prefix_tokens'):
            return int(self.backbone.num_prefix_tokens)
        storage_tokens = getattr(self.backbone, 'n_storage_tokens', None)
        if storage_tokens is None:
            storage_tokens = getattr(self.backbone, 'num_register_tokens', 0)
        return 1 + int(storage_tokens or 0)

    def _patch_grid(self, images):
        patch_size = getattr(self.backbone, 'patch_size', 16)
        if isinstance(patch_size, (tuple, list)):
            patch_h, patch_w = int(patch_size[0]), int(patch_size[1])
        else:
            patch_h = patch_w = int(patch_size)
        return images.shape[-2] // patch_h, images.shape[-1] // patch_w

    def _unwrap_tokens(self, prepared):
        if torch.is_tensor(prepared):
            return prepared
        if isinstance(prepared, (tuple, list)):
            for item in prepared:
                if torch.is_tensor(item):
                    return item
        if isinstance(prepared, dict):
            for key in ('x', 'tokens', 'last_hidden_state'):
                if key in prepared and torch.is_tensor(prepared[key]):
                    return prepared[key]
            for item in prepared.values():
                if torch.is_tensor(item):
                    return item
        raise TypeError(f'Cannot unwrap DINO token output of type {type(prepared)}')

    def _prepare_tokens_with_rope(self, images):
        try:
            prepared = self.backbone.prepare_tokens_with_masks(images, None)
        except TypeError:
            prepared = self.backbone.prepare_tokens_with_masks(images)
        tokens = self._unwrap_tokens(prepared)
        rope = None
        if hasattr(self.backbone, 'rope_embed'):
            grid_h, grid_w = self._patch_grid(images)
            try:
                rope = self.backbone.rope_embed(H=grid_h, W=grid_w)
            except TypeError:
                rope = self.backbone.rope_embed(grid_h, grid_w)
        return tokens, rope

    def _repeat_rope_for_query_context(self, rope):
        if rope is None or not torch.is_tensor(rope):
            return rope
        try:
            return rope.repeat_interleave(2, dim=-2)
        except RuntimeError:
            return rope

    def _run_blocks(self, tokens, rope):
        for block in self.backbone.blocks:
            if rope is None:
                tokens = block(tokens)
                continue
            try:
                tokens = block(tokens, rope=rope)
            except TypeError:
                try:
                    tokens = block(tokens, rope)
                except TypeError:
                    tokens = block(tokens)
        if hasattr(self.backbone, 'norm'):
            tokens = self.backbone.norm(tokens)
        return tokens

    def _forward_token_concat_features(self, query_images, context_images):
        query_tokens, query_rope = self._prepare_tokens_with_rope(query_images)
        context_tokens, _ = self._prepare_tokens_with_rope(context_images)
        prefix_tokens = self._prefix_token_count()
        pair_tokens = torch.cat([query_tokens[:, :prefix_tokens], query_tokens[:, prefix_tokens:], context_tokens[:, prefix_tokens:]], dim=1)
        pair_rope = self._repeat_rope_for_query_context(query_rope)
        pair_tokens = self._run_blocks(pair_tokens, pair_rope)
        return pair_tokens[:, 0]

    def _pool_backbone_features(self, images):
        features = self.backbone.forward_features(images)
        if isinstance(features, dict):
            if 'x_norm_clstoken' in features:
                return features['x_norm_clstoken']
            if 'x_norm_patchtokens' in features:
                return features['x_norm_patchtokens'].mean(dim=1)
            for item in features.values():
                if torch.is_tensor(item):
                    return item[:, 0] if item.ndim == 3 else item
        if torch.is_tensor(features):
            return features[:, 0] if features.ndim == 3 else features
        raise TypeError(f'Cannot pool DINO features of type {type(features)}')

    def _forward_feature_fusion(self, query_images, context_images):
        if context_images.ndim == 5:
            context_images = context_images[:, 0]
        query_features = self._pool_backbone_features(query_images)
        context_features = self._pool_backbone_features(context_images)
        self.active_fusion_mode = 'feature_fusion'
        return self.feature_fusion(torch.cat([query_features, context_features], dim=1))

    def _forward_topk_token_attention(self, query_images, context_images, context_scores=None):
        if context_images.ndim == 4:
            context_images = context_images.unsqueeze(1)
        batch_size, num_contexts, channels, height, width = context_images.shape
        query_repeated = query_images.unsqueeze(1).expand(-1, num_contexts, -1, -1, -1).reshape(batch_size * num_contexts, channels, height, width)
        context_flat = context_images.reshape(batch_size * num_contexts, channels, height, width)
        pair_features = self._forward_token_concat_features(query_repeated, context_flat).reshape(batch_size, num_contexts, self.feature_dim)
        query_summary = pair_features.mean(dim=1)
        attention_hidden = torch.tanh(self.context_proj(pair_features) + self.query_proj(query_summary).unsqueeze(1))
        attention_logits = self.context_attention(attention_hidden).squeeze(-1)
        if context_scores is not None:
            if context_scores.ndim == 1:
                context_scores = context_scores.unsqueeze(1)
            attention_logits = attention_logits + RETRIEVAL_SCORE_WEIGHT * context_scores.to(attention_logits.device, dtype=attention_logits.dtype)
        attention_weights = torch.softmax(attention_logits, dim=1)
        fused_features = torch.sum(pair_features * attention_weights.unsqueeze(-1), dim=1)
        fused_features = self.fusion_norm(fused_features)
        self.active_fusion_mode = 'topk_token_attention'
        return fused_features, attention_weights

    def forward(self, query_images, context_images, context_scores=None, return_features=False):
        try:
            fused_features, attention_weights = self._forward_topk_token_attention(query_images, context_images, context_scores=context_scores)
            logits = self.token_head(fused_features)
            if return_features:
                return logits, F.normalize(fused_features.float(), dim=1), attention_weights
            return logits
        except Exception as exc:
            if self.allow_feature_fallback:
                print(f'Top-k token attention failed, fallback to feature fusion: {type(exc).__name__}: {exc}')
                logits = self._forward_feature_fusion(query_images, context_images)
                if return_features:
                    return logits, F.normalize(logits.float(), dim=1), None
                return logits
            raise RuntimeError('Top-k token attention failed. Keep allow_feature_fallback=False for debugging.') from exc

class DinoV3GatedTopKContextClassifier(DinoV3FrozenSimilarityContextClassifier):
    def __init__(self, repo_dir, model_name, weights_path, num_classes, feature_dim=384, allow_feature_fallback=False):
        super().__init__(
            repo_dir=repo_dir,
            model_name=model_name,
            weights_path=weights_path,
            num_classes=num_classes,
            feature_dim=feature_dim,
            allow_feature_fallback=allow_feature_fallback,
        )
        self.query_norm = nn.LayerNorm(feature_dim)
        self.context_norm = nn.LayerNorm(feature_dim)
        self.final_norm = nn.LayerNorm(feature_dim)

        self.query_head = nn.Sequential(
            nn.Dropout(p=TOKEN_HEAD_DROPOUT),
            nn.Linear(feature_dim, num_classes),
        )

        self.fusion_gate = nn.Sequential(
            nn.Linear(feature_dim * 4, feature_dim),
            nn.GELU(),
            nn.Dropout(p=QUERY_CONTEXT_GATE_DROPOUT),
            nn.Linear(feature_dim, 1),
        )
        nn.init.constant_(self.fusion_gate[-1].bias, QUERY_CONTEXT_GATE_BIAS)
        self.active_fusion_mode = "query_plus_gated_topk_context"

    def forward(self, query_images, context_images, context_scores=None, return_features=False):
        try:
            context_features, attention_weights = self._forward_topk_token_attention(
                query_images,
                context_images,
                context_scores=context_scores,
            )
            query_features = self._pool_backbone_features(query_images)

            query_features = self.query_norm(query_features)
            context_features = self.context_norm(context_features)

            gate_input = torch.cat(
                [
                    query_features,
                    context_features,
                    query_features * context_features,
                    torch.abs(query_features - context_features),
                ],
                dim=1,
            )
            gate = torch.sigmoid(self.fusion_gate(gate_input))

            query_logits = self.query_head(query_features)
            context_logits = self.token_head(context_features)
            logits = query_logits + gate * context_logits

            fused_features = self.final_norm(query_features + gate * context_features)
            self.active_fusion_mode = "query_plus_gated_topk_context"

            if return_features:
                return logits, F.normalize(fused_features.float(), dim=1), attention_weights
            return logits

        except Exception as exc:
            if self.allow_feature_fallback:
                print(f"Gated fusion failed, fallback to parent model: {type(exc).__name__}: {exc}")
                return super().forward(query_images, context_images, context_scores=context_scores, return_features=return_features)
            raise




class TACSSelector(nn.Module):
    def __init__(self, temperature=1.0, normalize=True):
        super().__init__()
        self.temperature = float(temperature)
        self.normalize = bool(normalize)
        self.logit_scale = nn.Parameter(torch.tensor(1.0))

    def score(self, query_features, candidate_features):
        query = query_features.float()
        candidates = candidate_features.float()
        if self.normalize:
            query = F.normalize(query, dim=-1)
            candidates = F.normalize(candidates, dim=-1)
        scores = query @ candidates.T
        return scores * self.logit_scale.clamp(0.01, 100.0) / max(self.temperature, 1e-6)


class DinoV3TACSClassifier(DinoV3GatedTopKContextClassifier):
    def __init__(self, repo_dir, model_name, weights_path, num_classes, feature_dim=384, allow_feature_fallback=False):
        super().__init__(
            repo_dir=repo_dir,
            model_name=model_name,
            weights_path=weights_path,
            num_classes=num_classes,
            feature_dim=feature_dim,
            allow_feature_fallback=allow_feature_fallback,
        )
        self.selector = TACSSelector(
            temperature=TACS_SELECTOR_TEMPERATURE,
            normalize=True,
        )
        self.active_fusion_mode = 'tacs_full_pool_shared_backbone_selector'

    def selector_encode_images(self, images):
        return self._pool_backbone_features(images)

    def score_full_candidate_pool(self, query_images, candidate_feature_bank):
        query_features = self.selector_encode_images(query_images)
        selector_scores = self.selector.score(query_features, candidate_feature_bank.to(query_features.device))
        return selector_scores, query_features

    def select_full_pool(self, query_images, candidate_feature_bank, use_gumbel=True):
        selector_scores, query_features = self.score_full_candidate_pool(query_images, candidate_feature_bank)
        if self.training and use_gumbel:
            selection_weights = F.gumbel_softmax(
                selector_scores,
                tau=TACS_GUMBEL_TAU,
                hard=True,
                dim=1,
            )
            selected_indices = selection_weights.argmax(dim=1)
        else:
            selected_indices = selector_scores.argmax(dim=1)
            selection_weights = F.one_hot(selected_indices, num_classes=selector_scores.size(1)).to(selector_scores.dtype)
        details = {
            'selector_scores': selector_scores,
            'selection_weights': selection_weights,
            'selected_indices': selected_indices,
            'query_features': query_features,
            'selector_entropy': torch.distributions.Categorical(logits=selector_scores.float()).entropy().mean(),
        }
        self.active_fusion_mode = 'tacs_full_pool_shared_backbone_selector'
        return details

    def forward_selected_context(self, query_images, selected_context_images, return_features=False):
        context_images = selected_context_images.unsqueeze(1)
        return super().forward(query_images, context_images, context_scores=None, return_features=return_features)

    def forward_query_only(self, query_images):
        query_features = self._pool_backbone_features(query_images)
        if hasattr(self, 'query_norm') and hasattr(self, 'query_head'):
            query_features = self.query_norm(query_features)
            return self.query_head(query_features)
        return self.token_head(query_features)




# === M-TACS: TEXT/VISUAL FUSION ===
class MTACSModel(DinoV3TACSClassifier):
    """TACS selector and visual path with a gated CLIP text residual."""
    def __init__(self, *args, clip_dim=512, **kwargs):
        super().__init__(*args, **kwargs)
        self.text_projection = nn.Sequential(
            nn.Linear(clip_dim, FEATURE_DIM),
            nn.LayerNorm(FEATURE_DIM),
            nn.Dropout(0.10),
        )
        self.text_gate = nn.Sequential(
            nn.Linear(FEATURE_DIM * 2, FEATURE_DIM),
            nn.GELU(),
            nn.Linear(FEATURE_DIM, 1),
        )
        self.mtacs_norm = nn.LayerNorm(FEATURE_DIM)
        self.mtacs_classifier = nn.Sequential(
            nn.Dropout(TOKEN_HEAD_DROPOUT),
            nn.Linear(FEATURE_DIM, NUM_CLASSES),
        )
        nn.init.constant_(self.text_gate[-1].bias, -1.0)

    def _fuse_text(self, visual_features, text_embeddings, text_valid_mask):
        text_features = self.text_projection(text_embeddings.float())
        gate_input = torch.cat([visual_features.float(), text_features], dim=1)
        gate = torch.sigmoid(self.text_gate(gate_input))
        valid = text_valid_mask.to(gate.device, dtype=gate.dtype).view(-1, 1)
        gate = gate * valid
        fused = self.mtacs_norm(visual_features.float() + gate * text_features)
        return self.mtacs_classifier(fused), fused, gate.squeeze(1)

    def forward_selected_context_with_text(
        self, query_images, selected_context_images, text_embeddings, text_valid_mask,
        return_features=False,
    ):
        _, visual_features, attention = super().forward_selected_context(
            query_images, selected_context_images, return_features=True
        )
        logits, fused_features, gate = self._fuse_text(
            visual_features, text_embeddings, text_valid_mask
        )
        if return_features:
            return logits, F.normalize(fused_features.float(), dim=1), attention, gate
        return logits, gate

    def forward_query_with_text(
        self, query_images, text_embeddings, text_valid_mask, return_features=False
    ):
        query_features = self.query_norm(self._pool_backbone_features(query_images))
        logits, fused_features, gate = self._fuse_text(
            query_features, text_embeddings, text_valid_mask
        )
        if return_features:
            return logits, F.normalize(fused_features.float(), dim=1), gate
        return logits, gate


def build_candidate_image_bank(annotations, transform, device, dtype):
    dataset = SimpleImageDataset(annotations, transform, crop_bbox=USE_BBOX_CROP)
    loader = DataLoader(dataset, batch_size=RETRIEVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    images_bank = []
    labels = []
    fnames = []
    for images, labels_batch, fnames_batch, *_ in tqdm(loader, desc='build candidate image bank', leave=False):
        images_bank.append(images.to(device=device, dtype=dtype, non_blocking=True).contiguous())
        labels.extend([int(x) for x in labels_batch.tolist()])
        fnames.extend(fnames_batch)
    return torch.cat(images_bank, dim=0), torch.tensor(labels, dtype=torch.long, device=device), fnames


@torch.no_grad()
def refresh_candidate_feature_bank(network, candidate_image_bank, batch_size=CANDIDATE_FEATURE_BANK_BATCH_SIZE):
    was_training = network.training
    network.eval()
    features = []
    with torch.inference_mode():
        for start in tqdm(range(0, len(candidate_image_bank), batch_size), desc='refresh full candidate feature bank', leave=False):
            end = min(start + batch_size, len(candidate_image_bank))
            images = candidate_image_bank[start:end].to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=DEVICE.type == 'cuda'):
                feats = network.selector_encode_images(images)
            features.append(feats.float().detach().cpu())
    if was_training:
        network.train()
    return torch.cat(features, dim=0)


ensure_dinov3_assets()

model = MTACSModel(
    DINOV3_REPO_DIR,
    DINOV3_MODEL_NAME,
    resolve_dinov3_weights(),
    NUM_CLASSES,
    FEATURE_DIM,
    allow_feature_fallback=False,
    clip_dim=CLIP_EMBED_DIM,
).to(DEVICE)

candidate_image_bank, candidate_labels_tensor, candidate_fnames = build_candidate_image_bank(
    candidate_df,
    eval_transform,
    device=torch.device(CANDIDATE_IMAGE_BANK_DEVICE),
    dtype=CANDIDATE_IMAGE_BANK_DTYPE,
)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
backbone_params = [p for name, p in model.named_parameters() if name.startswith('backbone.')]
selector_params = [p for name, p in model.named_parameters() if name.startswith('selector.')]
head_params = [p for name, p in model.named_parameters() if not name.startswith('backbone.') and not name.startswith('selector.')]
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': BACKBONE_LR, 'weight_decay': BACKBONE_WEIGHT_DECAY, 'name': 'backbone'},
    {'params': selector_params, 'lr': TACS_SELECTOR_LR, 'weight_decay': TACS_SELECTOR_WEIGHT_DECAY, 'name': 'selector'},
    {'params': head_params, 'lr': HEAD_LR, 'weight_decay': HEAD_WEIGHT_DECAY, 'name': 'head'},
])

warmup_epochs = max(1, min(WARMUP_EPOCHS, EPOCHS))
main_epochs = max(1, EPOCHS - warmup_epochs)
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.2, end_factor=1.0, total_iters=warmup_epochs)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=main_epochs, eta_min=1e-7)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')

RUN_DIR = OUT_PUT_DIR / 'runs' / EXPERIMENT_MODE
RUN_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = RUN_DIR / 'mtacs_last.pt'
last_checkpoint_path = RUN_DIR / 'mtacs_last.pt'
best_checkpoint_path = RUN_DIR / 'mtacs_best.pt'

torch.save({
    'candidate_pool_positions': torch.as_tensor(candidate_pool_positions, dtype=torch.long),
    'seed': SEED,
    'candidate_pool_fraction': CANDIDATE_POOL_FRACTION,
    'use_bbox_crop': USE_BBOX_CROP,
    'selector_mode': 'full_candidate_pool_shared_backbone',
    'candidate_image_bank_device': CANDIDATE_IMAGE_BANK_DEVICE,
    'candidate_image_bank_dtype': str(CANDIDATE_IMAGE_BANK_DTYPE),
}, OUT_PUT_DIR / 'tacs_selector_cache.pt')

sample = train_dataset[0]
print('Saved TACS selector cache:', (OUT_PUT_DIR / 'tacs_selector_cache.pt').resolve())
print('Candidate image bank:', candidate_image_bank.shape, candidate_image_bank.device, candidate_image_bank.dtype)
print('Sample query tensor:', sample['query_image'].shape)
print('Candidate pool size used per query:', len(candidate_df))
print('Initial fusion mode:', model.active_fusion_mode)
print(model)


DINOv3 assets đã sẵn sàng từ Kaggle Input.


build candidate image bank:   0%|          | 0/7 [00:00<?, ?it/s]

Saved TACS selector cache: /kaggle/working/outputs/mtacs_sun397_visual_text_context/tacs_selector_cache.pt
Candidate image bank: torch.Size([3573, 3, 384, 384]) cuda:0 torch.float16
Sample query tensor: torch.Size([3, 384, 384])
Candidate pool size used per query: 3573
Initial fusion mode: tacs_full_pool_shared_backbone_selector
MTACSModel(
  (backbone): DinoVisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (rope_embed): RopePositionEmbedding()
    (blocks): ModuleList(
      (0-11): 12 x SelfAttentionBlock(
        (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (attn): SelfAttention(
          (qkv): LinearKMaskedBias(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1

/tmp/ipykernel_163/2166007692.py:513: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')


In [30]:
def _candidate_positions(indices):
    if not torch.is_tensor(indices):
        indices = torch.as_tensor(indices)
    return indices.to(
        device=candidate_image_bank.device,
        dtype=torch.long,
    ).reshape(-1)


def gather_bank_images(indices):
    positions = _candidate_positions(indices)
    return candidate_image_bank.index_select(0, positions).to(
        DEVICE, non_blocking=True
    )


def gather_bank_labels(indices):
    positions = _candidate_positions(indices).to(candidate_labels_tensor.device)
    return candidate_labels_tensor.index_select(0, positions).to(
        DEVICE, non_blocking=True
    )


def select_bank_images_from_weights(selection_weights):
    if selection_weights.ndim != 2 or selection_weights.shape[1] != len(candidate_image_bank):
        raise ValueError("selection_weights phải có shape [batch, candidate_pool_size]")
    selected_positions = selection_weights.argmax(dim=1)
    return gather_bank_images(selected_positions)

## Training

Run this cell once per `EXPERIMENT_MODE`. In M-TACS modes, the policy reward compares the query-plus-caption branch against query-plus-selected-image-plus-the-same-caption.

In [32]:
# === M-TACS: MODIFIED REWARD / TRAINING ===
from sklearn.metrics import f1_score

VALID_MODES = {"no_context", "original_tacs", "text_only", "mtacs", "mtacs_shuffled"}
if EXPERIMENT_MODE not in VALID_MODES:
    raise ValueError(f"EXPERIMENT_MODE must be one of {sorted(VALID_MODES)}")


def selected_text(batch, shuffled=False):
    ids = [str(value) for value in batch["image_id"]]
    lookup = shuffled_text_lookup if shuffled else text_embedding_lookup
    entries = [lookup[image_id] for image_id in ids]
    return (
        torch.stack([entry[0] for entry in entries]).to(DEVICE, non_blocking=True),
        torch.tensor([entry[1] for entry in entries], dtype=torch.bool, device=DEVICE),
    )


def mode_logits(network, mode, query_images, text_embeddings, text_valid, context_images=None, return_features=False):
    if mode == "no_context":
        logits = network.forward_query_only(query_images)
        return (logits, None, None) if return_features else (logits, None)
    if mode == "original_tacs":
        logits, features, attention = network.forward_selected_context(
            query_images, context_images, return_features=True
        )
        return (logits, features, None) if return_features else (logits, None)
    if mode == "text_only":
        logits, gate = network.forward_query_with_text(query_images, text_embeddings, text_valid)
        return (logits, None, gate) if return_features else (logits, gate)
    logits, features, attention, gate = network.forward_selected_context_with_text(
        query_images, context_images, text_embeddings, text_valid, return_features=True
    )
    return (logits, features, gate) if return_features else (logits, gate)


def mode_uses_visual_selector(mode):
    return mode in {"original_tacs", "mtacs", "mtacs_shuffled"}

def set_backbone_trainable(network, trainable=True):
    for parameter in network.backbone.parameters():
        parameter.requires_grad_(bool(trainable))
def supervised_contrastive_loss(features, labels, temperature=0.1):
    if temperature <= 0:
        raise ValueError("temperature must be positive")

    features = F.normalize(features.float(), dim=1)
    labels = labels.reshape(-1, 1)
    batch_size = features.shape[0]

    if batch_size < 2:
        return features.sum() * 0.0

    self_mask = torch.eye(batch_size, dtype=torch.bool, device=features.device)
    positive_mask = labels.eq(labels.T) & ~self_mask
    positive_counts = positive_mask.sum(dim=1)
    valid_anchors = positive_counts > 0

    if not valid_anchors.any():
        return features.sum() * 0.0

    logits = (features @ features.T) / float(temperature)
    logits = logits - logits.max(dim=1, keepdim=True).values.detach()
    log_denominator = torch.logsumexp(
        logits.masked_fill(self_mask, -torch.inf),
        dim=1,
        keepdim=True,
    )
    log_prob = (logits - log_denominator).masked_fill(~positive_mask, 0.0)
    mean_positive_log_prob = log_prob.sum(dim=1) / positive_counts.clamp_min(1)
    return -mean_positive_log_prob[valid_anchors].mean()        
def train_one_epoch_mtacs(network, loader, epoch, candidate_feature_bank):
    network.train()
    set_backbone_trainable(network, epoch > FREEZE_BACKBONE_EPOCHS)
    bank_device = candidate_feature_bank.to(DEVICE, non_blocking=True)
    sums = {key: 0.0 for key in ("loss", "ce_loss", "supcon_loss", "policy_loss", "reward", "entropy", "gate")}
    correct = total = selected_same = valid_text_count = 0
    predictions_all, labels_all = [], []
    for batch in tqdm(loader, desc=f"Epoch {epoch:03d} [{EXPERIMENT_MODE}]", leave=False):
        query = batch["query_image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)
        text, text_valid = selected_text(batch, shuffled=EXPERIMENT_MODE == "mtacs_shuffled")
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            details = None
            selected_images = None
            if mode_uses_visual_selector(EXPERIMENT_MODE):
                details = network.select_full_pool(query, bank_device, use_gumbel=True)
                selected_images = select_bank_images_from_weights(details["selection_weights"])
            logits, features, gate = mode_logits(
                network, EXPERIMENT_MODE, query, text, text_valid, selected_images, return_features=True
            )
            ce_loss = criterion(logits, labels)
            if features is not None:
                supcon_loss = supervised_contrastive_loss(features, labels, temperature=SUPCON_TEMPERATURE)
            else:
                supcon_loss = logits.sum() * 0.0

            policy_loss = logits.sum() * 0.0
            entropy = logits.new_zeros(())
            rewards = labels.new_zeros(labels.shape, dtype=torch.float32)
            if details is not None:
                selector_scores = details["selector_scores"].float()
                policy_dist = torch.distributions.Categorical(logits=selector_scores)
                policy_actions = policy_dist.sample()
                log_probs = policy_dist.log_prob(policy_actions)
                entropy = policy_dist.entropy().mean()
                policy_context_images = gather_bank_images(policy_actions)
                # Keep identical caption and mask on both reward branches.
                # The reward therefore measures visual-context utility only.
                with torch.no_grad():
                    if EXPERIMENT_MODE == "original_tacs":
                        baseline_logits = network.forward_query_only(query)
                        context_logits = network.forward_selected_context(query, policy_context_images, return_features=False)
                    else:
                        baseline_logits, _ = network.forward_query_with_text(query, text, text_valid)
                        context_logits, _ = network.forward_selected_context_with_text(
                            query, policy_context_images, text, text_valid
                        )
                    baseline_losses = F.cross_entropy(baseline_logits.float(), labels, reduction="none")
                    context_losses = F.cross_entropy(context_logits.float(), labels, reduction="none")
                    rewards = (baseline_losses - context_losses).detach()
                    reward_std = rewards.std(unbiased=False).clamp_min(1e-6)
                    advantages = ((rewards - rewards.mean()) / reward_std).detach()
                policy_loss = -(log_probs * advantages).mean()

            loss = ce_loss + SUPCON_WEIGHT * supcon_loss
            if details is not None:
                loss = loss + LAMBDA_POLICY * policy_loss - TACS_ENTROPY_WEIGHT * entropy

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(network.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        batch_size = int(labels.size(0))
        predicted = logits.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += batch_size
        predictions_all.extend(predicted.detach().cpu().tolist())
        labels_all.extend(labels.detach().cpu().tolist())
        valid_text_count += text_valid.sum().item()
        if details is not None:
            selected_same += (gather_bank_labels(details["selected_indices"]) == labels).sum().item()
        for key, value in (("loss", loss), ("ce_loss", ce_loss), ("supcon_loss", supcon_loss),
                           ("policy_loss", policy_loss), ("reward", rewards.mean()), ("entropy", entropy)):
            sums[key] += float(value.detach().item()) * batch_size
        if gate is not None:
            sums["gate"] += float(gate.detach().mean().item()) * batch_size

    denominator = max(total, 1)
    return {
        "loss": sums["loss"] / denominator,
        "ce_loss": sums["ce_loss"] / denominator,
        "supcon_loss": sums["supcon_loss"] / denominator,
        "policy_loss": sums["policy_loss"] / denominator,
        "reward": sums["reward"] / denominator,
        "selector_entropy": sums["entropy"] / denominator,
        "accuracy": correct / denominator,
        "macro_f1": float(f1_score(labels_all, predictions_all, average="macro", zero_division=0)),
        "selected_same_class_rate": selected_same / denominator,
        "mean_gate": sums["gate"] / denominator,
        "valid_caption_rate": valid_text_count / denominator,
    }


@torch.inference_mode()
def evaluate_mtacs(network, loader, candidate_feature_bank, mode, desc="validation"):
    network.eval()
    bank_device = candidate_feature_bank.to(DEVICE, non_blocking=True)
    total_loss = correct = total = selected_same = valid_count = 0
    gate_sum = 0.0
    labels_all, predictions_all, rows = [], [], []
    for batch in tqdm(loader, desc=desc, leave=False):
        query = batch["query_image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)
        text, text_valid = selected_text(batch, shuffled=mode == "mtacs_shuffled")
        details = network.select_full_pool(query, bank_device, use_gumbel=False) if mode_uses_visual_selector(mode) else None
        context = gather_bank_images(details["selected_indices"]) if details is not None else None
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            logits, gate = mode_logits(network, mode, query, text, text_valid, context)
            loss = criterion(logits, labels)
        preds = logits.argmax(dim=1)
        count = int(labels.size(0))
        total += count
        correct += (preds == labels).sum().item()
        total_loss += float(loss.item()) * count
        labels_all.extend(labels.cpu().tolist())
        predictions_all.extend(preds.cpu().tolist())
        valid_count += text_valid.sum().item()
        if gate is not None:
            gate_sum += float(gate.mean().item()) * count
        for j in range(count):
            pos = int(details["selected_indices"][j]) if details is not None else -1
            row = {
                "image_id": str(batch["image_id"][j]), "fname": str(batch["fname"][j]),
                "ground_truth": int(labels[j]), "prediction": int(preds[j]),
                "correct": bool(labels[j] == preds[j]),
                "caption": str(caption_text_by_id.get(str(batch["image_id"][j]), "")),
                "caption_valid": bool(text_valid[j]), "gate_value": float(gate[j]) if gate is not None else np.nan,
                "selected_context_position": pos,
                "selected_context_id": str(candidate_df.iloc[pos]["image_id"]) if pos >= 0 else "",
                "selected_context_fname": str(candidate_df.iloc[pos]["fname"]) if pos >= 0 else "",
                "selector_probability": float(torch.softmax(details["selector_scores"][j].float(), dim=0)[pos]) if pos >= 0 else np.nan,
            }
            rows.append(row)
            if details is not None:
                selected_same += int(candidate_labels_tensor[pos].item() == int(labels[j]))
    denominator = max(total, 1)
    return {
        "loss": total_loss / denominator,
        "accuracy": correct / denominator,
        "macro_f1": float(f1_score(labels_all, predictions_all, average="macro", zero_division=0)),
        "selected_same_class_rate": selected_same / denominator,
        "mean_gate": gate_sum / denominator,
        "valid_caption_rate": valid_count / denominator,
        "mean_gate_correct": float(np.mean([float(row["gate_value"]) for row in rows if row["correct"] and pd.notna(row["gate_value"])])) if any(row["correct"] and pd.notna(row["gate_value"]) for row in rows) else 0.0,
        "mean_gate_incorrect": float(np.mean([float(row["gate_value"]) for row in rows if not row["correct"] and pd.notna(row["gate_value"])])) if any(not row["correct"] and pd.notna(row["gate_value"]) for row in rows) else 0.0,
        "predictions": pd.DataFrame(rows),
    }


def build_mtacs_checkpoint(epoch, train_metrics, val_metrics, history, best_val_loss, best_epoch):
    state = model.state_dict()
    return {
        "epoch": epoch,
        "model_state_dict": state,
        "task_model_state_dict": {k: v for k, v in state.items() if not k.startswith("selector.")},
        "selector_state_dict": model.selector.state_dict(),
        "text_projection_state_dict": model.text_projection.state_dict(),
        "text_gate_state_dict": model.text_gate.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "class_names": class_names,
        "metrics": {"train": train_metrics, "val": val_metrics, "best_val_loss": best_val_loss, "best_epoch": best_epoch},
        "history": history,
        "seed": SEED,
        "clip_model_name": CLIP_MODEL_NAME,
        "vlm_model": VLM_MODEL,
        "prompt_version": PROMPT_VERSION,
        "config": {
            "dataset": "SUN397", "method": "M-TACS", "experiment_mode": EXPERIMENT_MODE,
            "selector_mode": "full_candidate_pool_shared_backbone", "candidate_pool_fraction": CANDIDATE_POOL_FRACTION,
            "candidate_pool_size": len(candidate_df), "tacs_gumbel_tau": TACS_GUMBEL_TAU,
            "tacs_policy_weight": LAMBDA_POLICY, "tacs_entropy_weight": TACS_ENTROPY_WEIGHT,
            "clip_model_name": CLIP_MODEL_NAME, "clip_embedding_dim": CLIP_EMBED_DIM,
            "text_fusion": "gated_additive_cls_representation", "image_size": IMAGE_SIZE,
            "dino_model": DINOV3_MODEL_NAME, "seed": SEED,
        },
        "candidate_pool_positions": torch.as_tensor(candidate_pool_positions, dtype=torch.long),
    }


RESUME_FROM_LAST_CHECKPOINT = False
history = []
best_val_loss = float("inf")
best_epoch = 0
bad_epochs = 0
candidate_feature_bank = refresh_candidate_feature_bank(model, candidate_image_bank)
start_epoch = 1
if RESUME_FROM_LAST_CHECKPOINT and last_checkpoint_path.is_file():
    resume_state = torch.load(last_checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(resume_state["model_state_dict"])
    optimizer.load_state_dict(resume_state["optimizer_state_dict"])
    scheduler.load_state_dict(resume_state["scheduler_state_dict"])
    scaler.load_state_dict(resume_state["scaler_state_dict"])
    history = resume_state.get("history", [])
    previous = resume_state.get("metrics", {})
    best_val_loss = float(previous.get("best_val_loss", float("inf")))
    best_epoch = int(previous.get("best_epoch", 0))
    start_epoch = int(resume_state["epoch"]) + 1
    print("Resuming at epoch", start_epoch)

actual_config = {
    "dataset": "SUN397", "experiment_mode": EXPERIMENT_MODE, "seed": SEED,
    "candidate_pool_fraction": CANDIDATE_POOL_FRACTION, "candidate_pool_size": len(candidate_df),
    "val_fraction": VAL_FRACTION, "dino_model": DINOV3_MODEL_NAME,
    "clip_model": CLIP_MODEL_NAME, "vlm_model": VLM_MODEL,
    "caption_model": VLM_MODEL, "prompt_version": PROMPT_VERSION,
    "clip_embedding_dim": CLIP_EMBED_DIM, "fusion": "gated_additive_cls_representation",
    "gumbel_tau": TACS_GUMBEL_TAU, "policy_weight": LAMBDA_POLICY,
}
(RUN_DIR / "mtacs_config.json").write_text(json.dumps(actual_config, indent=2), encoding="utf-8")

for epoch in range(start_epoch, EPOCHS + 1):
    candidate_feature_bank = refresh_candidate_feature_bank(model, candidate_image_bank)
    train_metrics = train_one_epoch_mtacs(model, train_loader, epoch, candidate_feature_bank)
    val_result = evaluate_mtacs(model, val_loader, candidate_feature_bank, EXPERIMENT_MODE, desc=f"Epoch {epoch:03d} [val]")
    val_metrics = {key: value for key, value in val_result.items() if key != "predictions"}
    scheduler.step()
    is_best = val_metrics["loss"] < best_val_loss - 1e-6
    if is_best:
        best_val_loss, best_epoch, bad_epochs = val_metrics["loss"], epoch, 0
    else:
        bad_epochs += 1
    record = {"epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()},
              **{f"val_{k}": v for k, v in val_metrics.items()}, "best_epoch": best_epoch}
    history.append(record)
    payload = build_mtacs_checkpoint(epoch, train_metrics, val_metrics, history, best_val_loss, best_epoch)
    torch.save(payload, last_checkpoint_path)
    if is_best:
        torch.save(payload, best_checkpoint_path)
    pd.DataFrame(history).to_csv(RUN_DIR / "training_history.csv", index=False)
    print(f"Epoch {epoch:03d}/{EPOCHS} | train loss={train_metrics['loss']:.4f} acc={train_metrics['accuracy']:.4f} "
          f"reward={train_metrics['reward']:.4f} policy={train_metrics['policy_loss']:.4f} "
          f"gate={train_metrics['mean_gate']:.3f} | val loss={val_metrics['loss']:.4f} "
          f"acc={val_metrics['accuracy']:.4f} f1={val_metrics['macro_f1']:.4f}")
    if bad_epochs >= PATIENCE:
        print(f"Early stopping at epoch {epoch}; best epoch={best_epoch}")
        break

print("Best checkpoint:", best_checkpoint_path)


refresh full candidate feature bank:   0%|          | 0/28 [00:00<?, ?it/s]

refresh full candidate feature bank:   0%|          | 0/28 [00:00<?, ?it/s]

Epoch 001 [mtacs]:   0%|          | 0/280 [00:00<?, ?it/s]

Epoch 001 [val]:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 001/100 | train loss=6.1496 acc=0.0046 reward=0.0288 policy=0.0009 gate=0.031 | val loss=5.9289 acc=0.0096 f1=0.0093


refresh full candidate feature bank:   0%|          | 0/28 [00:00<?, ?it/s]

... (168 outputs left out)

refresh full candidate feature bank:   0%|          | 0/28 [00:00<?, ?it/s]

Epoch 044 [mtacs]:   0%|          | 0/280 [00:00<?, ?it/s]

Epoch 044 [val]:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 044/100 | train loss=1.2206 acc=0.9612 reward=0.2275 policy=-0.0001 gate=0.017 | val loss=1.6987 acc=0.7793 f1=0.7767
Early stopping at epoch 44; best epoch=34
Best checkpoint: /kaggle/working/outputs/mtacs_sun397_visual_text_context/runs/mtacs/mtacs_best.pt


## Test metrics and ablation comparison

This cell loads the best validation checkpoint, reports test loss/accuracy/macro-F1, and saves per-image predictions and error-analysis artifacts.

In [ ]:
# === M-TACS: TEST EVALUATION / ERROR ANALYSIS ===
checkpoint = torch.load(best_checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
candidate_feature_bank = refresh_candidate_feature_bank(model, candidate_image_bank)
test_result = evaluate_mtacs(model, test_loader, candidate_feature_bank, EXPERIMENT_MODE, desc="official test")
test_predictions = test_result.pop("predictions")
metrics = {f"test_{key}": value for key, value in test_result.items()}
caption_audit = caption_df[caption_df["split"].astype(str) == "test"]
metrics["test_leakage_detected_rate"] = pd.to_numeric(caption_audit["leakage_detected"], errors="coerce").fillna(False).astype(bool).mean()
metrics["test_caption_retry_rate"] = (pd.to_numeric(caption_audit["retry_count"], errors="coerce").fillna(0) > 0).mean()
test_predictions.to_csv(RUN_DIR / "mtacs_predictions.csv", index=False)
with (RUN_DIR / "metrics.json").open("w", encoding="utf-8") as f:
    json.dump({"method": EXPERIMENT_MODE, **metrics, "best_epoch": checkpoint["epoch"],
               "candidate_pool_size": len(candidate_df), "valid_caption_rate": metrics["test_valid_caption_rate"]}, f, indent=2)
print(f"Method: {EXPERIMENT_MODE}")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")
print("Predictions:", RUN_DIR / "mtacs_predictions.csv")
display(test_predictions.head())


def build_ablation_summary(base_dir=OUT_PUT_DIR / "runs"):
    rows = []
    for mode in ("no_context", "original_tacs", "text_only", "mtacs", "mtacs_shuffled"):
        metrics_path = base_dir / mode / "metrics.json"
        if metrics_path.is_file():
            saved = json.loads(metrics_path.read_text(encoding="utf-8"))
            rows.append({"method": mode, "accuracy": saved.get("test_accuracy"),
                         "macro_f1": saved.get("test_macro_f1"), "loss": saved.get("test_loss"),
                         "mean_gate": saved.get("test_mean_gate"),
                         "valid_caption_rate": saved.get("test_valid_caption_rate")})
    summary = pd.DataFrame(rows)
    if not summary.empty:
        summary.to_csv(OUT_PUT_DIR / "ablation_summary.csv", index=False)
    return summary


display(build_ablation_summary())


def write_tacs_vs_mtacs_error_analysis(base_dir=OUT_PUT_DIR / "runs"):
    tacs_path = base_dir / "original_tacs" / "mtacs_predictions.csv"
    mtacs_path = base_dir / "mtacs" / "mtacs_predictions.csv"
    if not tacs_path.is_file() or not mtacs_path.is_file():
        print("Run original_tacs and mtacs first to compare per-image wins/losses.")
        return None
    tacs = pd.read_csv(tacs_path).rename(columns={"prediction": "tacs_prediction", "correct": "tacs_correct"})
    mtacs = pd.read_csv(mtacs_path).rename(columns={"prediction": "mtacs_prediction", "correct": "mtacs_correct"})
    merged = tacs[["image_id", "ground_truth", "tacs_prediction", "tacs_correct"]].merge(
        mtacs[["image_id", "mtacs_prediction", "mtacs_correct", "caption", "caption_valid",
               "selected_context_id", "selector_probability", "gate_value"]], on="image_id", how="inner"
    )
    merged["case_type"] = np.select(
        [~merged.tacs_correct & merged.mtacs_correct, merged.tacs_correct & ~merged.mtacs_correct,
         merged.tacs_correct & merged.mtacs_correct],
        ["TACS wrong -> M-TACS correct", "TACS correct -> M-TACS wrong", "both correct"], default="both wrong"
    )
    out = base_dir / "mtacs_vs_tacs_error_analysis.csv"
    merged.to_csv(out, index=False)
    print("Saved:", out)
    return merged


error_analysis = write_tacs_vs_mtacs_error_analysis()


refresh full candidate feature bank:   0%|          | 0/28 [00:00<?, ?it/s]

official test:   0%|          | 0/311 [00:00<?, ?it/s]

Method: mtacs
test_loss: 1.7653
test_accuracy: 0.7612
test_macro_f1: 0.7602
test_selected_same_class_rate: 0.6437
test_mean_gate: 0.0163
test_valid_caption_rate: 0.6055
test_mean_gate_correct: 0.0168
test_mean_gate_incorrect: 0.0148
test_leakage_detected_rate: 0.3929
test_caption_retry_rate: 0.0000
Predictions: /kaggle/working/outputs/mtacs_sun397_visual_text_context/runs/mtacs/mtacs_predictions.csv


,image_id,fname,ground_truth,prediction,correct,caption,caption_valid,gate_value,selected_context_position,selected_context_id,selected_context_fname,selector_probability
0,a/abbey/sun_ajkqrqitspwywirx.jpg,sun_ajkqrqitspwywirx.jpg,0,0,True,The image shows a large ornate cathedral with,True,0.019531,0,a/abbey/sun_aakbdcgfpksytcwj.jpg,sun_aakbdcgfpksytcwj.jpg,0.000701
1,a/abbey/sun_ajhtswxgrqbeiikc.jpg,sun_ajhtswxgrqbeiikc.jpg,0,0,True,The image shows an old dilapidated church in,True,0.017242,2151,m/monastery/outdoor/sun_asadwdtegcubkjss.jpg,sun_asadwdtegcubkjss.jpg,0.000426
2,a/abbey/sun_ahebfjooupcqbjht.jpg,sun_ahebfjooupcqbjht.jpg,0,231,False,The image is a photograph of a large,True,0.013901,2141,m/moat/water/sun_bjlukomljlmikwdy.jpg,sun_bjlukomljlmikwdy.jpg,0.000407
3,a/abbey/sun_amdwiijwnagslgef.jpg,sun_amdwiijwnagslgef.jpg,0,84,False,The image shows the interior of a large,False,0.000000,80,a/apse/indoor/sun_bxflqprgluymhlhh.jpg,sun_bxflqprgluymhlhh.jpg,0.000534
4,a/abbey/sun_awzqxzqukmbujwzm.jpg,sun_awzqxzqukmbujwzm.jpg,0,0,True,The image is a photograph of a tall,True,0.009636,0,a/abbey/sun_aakbdcgfpksytcwj.jpg,sun_aakbdcgfpksytcwj.jpg,0.000528


,method,accuracy,macro_f1,loss,mean_gate,valid_caption_rate
0,mtacs,0.761159,0.760189,1.765304,0.016345,0.605542


Run original_tacs and mtacs first to compare per-image wins/losses.
